# 06 — Streamlit Demo (live HEMT-CLIP inference + XAI)

Launches `app/streamlit_app.py` on Colab and exposes it via an **ngrok tunnel** so the demo can be opened in a browser from anywhere (e.g. during viva). Per Blueprint §11.3, this is the recommended deployment path for the viva — Streamlit Community Cloud has no GPU.

**What the demo does:**
- Pick a test sample from the dropdown (3 real + 3 fake test posts) OR upload your own image + title.
- Live cross-attention heatmap (one forward pass, < 1 s on T4).
- Pre-computed SHAP attribution for test samples (live SHAP would block the UI).
- CLIP text-image alignment (α) with verbal interpretation.
- "How it works" tab with architecture summary and test-set headline numbers.

**Prerequisites:**
- An ngrok account (free) + auth token. Sign up at https://dashboard.ngrok.com/. Free tier is fine — we use one tunnel for the demo session.
- A `hemt_clip` checkpoint on Drive (the canonical v4 seed=42 ckpt, same as notebooks 04–05).
- The XAI artefacts from notebook 05 (`outputs/xai/shap/*.png` and `shap_manifest.json`) — auto-recovered below if missing.

## Setup
Same idempotent bootstrap as notebooks 02–05. Mount Drive (Account B in the OAuth popup), pull repo, install deps (including `streamlit` + `pyngrok`), copy HDF5 to local SSD.

In [ ]:
# Bootstrap — idempotent. Safe to re-run on a fresh or warm Colab runtime.
import os, sys, subprocess, shutil

os.environ['USE_FLAX'] = 'FALSE'
os.environ['USE_TF'] = 'FALSE'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
REPO_URL = 'https://github.com/staharizvi/hemt-clip-fnd.git'
REPO_DIR = '/content/hemt-clip-fnd'
H5_DRIVE = '/content/drive/MyDrive/hemt-clip-fnd/data/fakeddit.h5'
H5_LOCAL = '/content/fakeddit.h5'

if IN_COLAB:
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')

    if os.path.exists(os.path.join(REPO_DIR, '.git')):
        print('Repo present — pulling latest…')
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    else:
        print('Cloning repo…')
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)

    subprocess.run(['pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
    subprocess.run(['pip', 'uninstall', '-y', '-q', 'jax', 'jaxlib', 'flax'], check=False)

    if not os.path.exists(H5_LOCAL):
        if os.path.exists(H5_DRIVE):
            print(f'Copying {H5_DRIVE} -> {H5_LOCAL}…')
            shutil.copy(H5_DRIVE, H5_LOCAL)
        else:
            print(f'WARNING: {H5_DRIVE} not found.')
    else:
        print(f'h5 already at {H5_LOCAL}.')

    os.chdir(REPO_DIR)

print('\ncwd:', os.getcwd())
print('h5 :', H5_LOCAL, 'exists:', os.path.exists(H5_LOCAL))

## Point the app at the local HDF5
The Streamlit app reads `cfg.data.hdf5_path` from `configs/base.yaml` to load the test-sample dropdown. Same yaml patch as nb 03/04/05.

In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
if cfg['data']['hdf5_path'] != '/content/fakeddit.h5':
    cfg['data']['hdf5_path'] = '/content/fakeddit.h5'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Patched cfg.data.hdf5_path -> /content/fakeddit.h5')
else:
    print('cfg already points at local HDF5.')
print('checkpoints :', cfg['checkpointing']['dir'])

## Ensure SHAP artefacts exist

The Streamlit app shows pre-computed SHAP attribution when the user picks a test sample. If notebook 05's outputs aren't in this Colab session (`/content/` gets wiped between runtimes), the next cell will run `training.evaluate` and `explainability.shap_text` to produce them.

`outputs/xai/attention/` is **not** auto-regenerated — the app does attention live, so the precomputed attention files are not needed for the demo.

In [ ]:
from pathlib import Path
import subprocess

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

preds_text  = Path('outputs/eval/preds_text_only.npz')
shap_manif  = Path('outputs/xai/shap/shap_manifest.json')

if not preds_text.exists():
    print('preds_text_only.npz missing — running training.evaluate (~3 min on T4)…')
    rc = subprocess.run([sys.executable, '-m', 'training.evaluate', '--split', 'test']).returncode
    if rc != 0:
        raise RuntimeError(f'training.evaluate failed rc={rc}')

if not shap_manif.exists():
    print('shap_manifest.json missing — running explainability.shap_text (~30 s on T4)…')
    # Reuse the same discovery as notebook 05.
    from training.evaluate import discover_checkpoints
    ckpts = discover_checkpoints(Path(cfg['checkpointing']['dir']))
    rc = subprocess.run([
        sys.executable, '-m', 'explainability.shap_text',
        '--checkpoint', str(ckpts['text_only']),
        '--preds-npz', 'outputs/eval/preds_text_only.npz',
        '--n-samples', '30',
    ]).returncode
    if rc != 0:
        raise RuntimeError(f'explainability.shap_text failed rc={rc}')

print('SHAP manifest:', shap_manif.exists())
if shap_manif.exists():
    import json
    m = json.loads(shap_manif.read_text(encoding='utf-8'))
    print(f'  {len(m)} samples covered.')

## Point the app at the canonical hemt_clip checkpoint
Set `HEMT_CLIP_CKPT` so `streamlit_app.py`'s `discover_ckpt()` skips its directory-walk fallback. We use the same auto-discovery logic as notebook 04/05 to pick the canonical v4 seed=42 ckpt (latest non-`_seed*` hemt_clip best.pt).

In [ ]:
from training.evaluate import discover_checkpoints

ckpts = discover_checkpoints(Path(cfg['checkpointing']['dir']))
HEMT_CLIP_CKPT = str(ckpts['hemt_clip'])
os.environ['HEMT_CLIP_CKPT'] = HEMT_CLIP_CKPT
print('HEMT_CLIP_CKPT =', HEMT_CLIP_CKPT)

## ngrok authentication

ngrok exposes the local Streamlit port (8501) on a public HTTPS URL. Free tier is sufficient — one tunnel per session, expires when the cell process ends.

**Get your token:** https://dashboard.ngrok.com/get-started/your-authtoken (free signup).

Either:
- Set the env var `NGROK_AUTHTOKEN` before running this notebook, or
- Run the cell below and paste the token into the `getpass` prompt (won't appear in the notebook output).

In [ ]:
import getpass
from pyngrok import ngrok, conf

token = os.environ.get('NGROK_AUTHTOKEN')
if not token:
    token = getpass.getpass('Paste ngrok auth token: ').strip()

ngrok.set_auth_token(token)
conf.get_default().monitor_thread = False  # quiet logging

# Tear down any leftover tunnels from a previous run in this kernel.
for t in ngrok.get_tunnels():
    try:
        ngrok.disconnect(t.public_url)
    except Exception:
        pass
ngrok.kill()
print('ngrok auth set + previous tunnels cleared.')

## Launch Streamlit + open the tunnel

Starts the app as a background subprocess on port 8501 (default), waits a few seconds for it to bind, then opens an ngrok HTTPS tunnel pointing at that port. The public URL prints below — click it to open the demo in a new tab.

**During viva:** copy the URL, paste it into the examiner's browser. The app stays live until you interrupt the next cell or the Colab runtime ends.

In [ ]:
import subprocess, time
from pyngrok import ngrok

PORT = 8501

# Kill any stale streamlit before launching a fresh one.
subprocess.run(['pkill', '-f', 'streamlit run'], check=False)
time.sleep(1)

log_path = '/content/streamlit.log'
with open(log_path, 'w') as logf:
    proc = subprocess.Popen(
        ['streamlit', 'run', 'app/streamlit_app.py',
         '--server.port', str(PORT),
         '--server.headless', 'true',
         '--browser.gatherUsageStats', 'false'],
        stdout=logf, stderr=subprocess.STDOUT, env={**os.environ},
    )
print(f'streamlit pid={proc.pid}, logs -> {log_path}')

# Wait for streamlit to bind (model loading happens lazily on first request, so a few s is enough).
for i in range(20):
    time.sleep(1)
    if subprocess.run(['curl', '-fs', f'http://127.0.0.1:{PORT}/'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0:
        print(f'streamlit is up after {i+1}s')
        break
else:
    print('streamlit did not come up in 20s — check /content/streamlit.log')

tunnel = ngrok.connect(PORT, proto='http', bind_tls=True)
print()
print('━' * 60)
print(f'  HEMT-CLIP demo URL:  {tunnel.public_url}')
print('━' * 60)

## Inspect the Streamlit logs (optional)
If the demo behaves strangely, tail the log file the launch cell writes. Useful for spotting first-request model-load errors or out-of-memory issues.

In [ ]:
!tail -n 40 /content/streamlit.log

## Shut down

Run when finished — closes the tunnel and stops the Streamlit process. Cell does *not* run unless you execute it; the app stays live in the background through any other notebook activity until then.

In [ ]:
from pyngrok import ngrok

for t in ngrok.get_tunnels():
    try:
        ngrok.disconnect(t.public_url)
    except Exception:
        pass
ngrok.kill()
subprocess.run(['pkill', '-f', 'streamlit run'], check=False)
print('Demo shut down.')

## Viva talking points (demo-specific)

Two dry-runs before viva day are non-negotiable per Blueprint §12, Day 14.

**Live capabilities to highlight:**
- *Attention heatmap is real-time.* One forward pass, < 1 s on GPU. The architecture exposes per-head attention from the fusion module; we average over heads and overlay on the input image. This is the **unique** capability of the cross-attention architecture — concat fusion produces no comparable visualisation.
- *α gives a quick sanity check.* Low α (< 0.15) usually means the image and title don't semantically match in CLIP space — common on deliberately misleading posts. Moderate (0.15–0.30) is typical for Fakeddit. High (> 0.30) is the well-aligned baseline.
- *Pre-computed SHAP* on test samples lets us discuss the methodology figures from report §6.4 (Kristallnacht error, tree-on-rock success, mad max method divergence) on the actual outputs the examiner can scroll through.

**Backup if Colab disconnects mid-viva:**
1. Bring a screen-recorded video of the demo (record once, replay if needed).
2. Keep the static figures from notebook 04 + 05 open in tabs (they're in the report appendix anyway).
3. Have the report's quantitative results section ready — answers any "how does it perform?" question without needing the live demo.

**Anticipated examiner questions on the demo:**
- *"Why does the attention heatmap look diffuse on this example?"* → Because the image content itself is distributed (text-heavy, no clear subject). Honest limit, covered in §6.4 Finding 1.
- *"Why is SHAP precomputed, not live?"* → 1000 perturbations × 1 s each = 17 min per attribution. Pre-computation on a 30-sample fixed set makes the demo interactive without sacrificing methodology.
- *"Can it handle adversarial / deepfake images?"* → Out of scope for FYP. Future work. The model was trained on Fakeddit's natural fake distribution (Photoshops, satire, mislabeled photos), not crafted adversarial examples.